# Cross-Source Sensitive Data Leakage (BigID × Sentinel Data Lake)

**Question:** Which users touched the same classification across *multiple* data sources (e.g. PHI in both Snowflake AND Box)?
Multi-source access patterns flag data exfiltration risk.

In [ ]:
# === Setup: connect to the Microsoft Sentinel data lake ===
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

data_provider = MicrosoftSentinelProvider(spark)

WORKSPACE = "<YOUR_SENTINEL_WORKSPACE_NAME>"
TABLE = "BigIDDSPMCatalog_CL"

# Pull last 30 days of BigID catalog rows
df = data_provider.read_table(TABLE, WORKSPACE)
df = df.filter(F.col("TimeGenerated") > F.expr("current_timestamp() - INTERVAL 30 DAYS"))
df.printSchema()
print("Row count:", df.count())


## Find cross-source classification access

In [ ]:
from pyspark.sql.functions import col, countDistinct

sensitive = df.filter(
    (col("Classification").contains("PHI")) |
    (col("Classification").contains("GDPR")) |
    (col("Classification").contains("Restricted")) |
    (col("Classification").contains("Confidential"))
).filter(col("UserName").isNotNull())

user_class_sources = (
    sensitive.groupBy("UserName", "Classification")
    .agg(countDistinct("AssetSource").alias("SourceCount"),
         F.collect_set("AssetSource").alias("Sources"),
         F.countDistinct("AssetID").alias("Assets"))
    .filter(col("SourceCount") >= 2)
    .orderBy(F.desc("SourceCount"), F.desc("Assets"))
)
user_class_sources.show(50, truncate=False)


## Edges: User → AssetSource (limited to multi-source users)

In [ ]:
multi = user_class_sources.select("UserName").distinct()
result = (
    sensitive.join(multi, "UserName")
    .select(col("UserName").alias("User"), col("AssetSource").alias("Source"))
    .distinct()
    .limit(120)
)
result.show(20, truncate=False)


## Visualize

In [ ]:
# === Visualize as a graph ===
import matplotlib.pyplot as plt
import networkx as nx

pdf = result.toPandas()
print(f"Edges to draw: {len(pdf)}")
display(pdf.head(50))

G = nx.DiGraph()
for _, row in pdf.iterrows():
    src = str(row.iloc[0])
    dst = str(row.iloc[1])
    G.add_edge(src, dst)

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=0.6)
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[n for n in G.nodes if n in pdf.iloc[:, 0].values],
    node_color="#1f77b4", node_size=900, alpha=0.85,
)
nx.draw_networkx_nodes(
    G, pos,
    nodelist=[n for n in G.nodes if n in pdf.iloc[:, 1].values and n not in pdf.iloc[:, 0].values],
    node_color="#d62728", node_size=900, alpha=0.85,
)
nx.draw_networkx_edges(G, pos, arrows=True, edge_color="#888", alpha=0.6, width=1.2)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title("Cross-Source Leakage — Users → Multiple Sensitive Data Sources", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()
